# 01 — Découverte du fichier DVF 2025

Objectif : comprendre le fichier brut avant de choisir une observation pour le futur modèle. Ici, on inspecte la structure, les valeurs manquantes et les répétitions de prix. **Aucune donnée n’est nettoyée et aucun modèle n’est entraîné dans cette étape.**

Le fichier est volumineux : l’analyse globale ci-dessous le lit par blocs (`chunksize`) afin de limiter la mémoire utilisée.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("data/raw/ValeursFoncieres-2025.txt")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} introuvable. Lance Jupyter depuis la racine du dépôt."
    )

print(f"Fichier : {DATA_PATH}")
print(f"Taille : {DATA_PATH.stat().st_size / 1024**2:.1f} Mio")

## 1. Examiner le format et les colonnes

Le séparateur du fichier est la barre verticale (`|`). Les champs restent d’abord en texte : cela préserve notamment les codes de département et les nombres avec virgule décimale.

In [ ]:
columns = pd.read_csv(DATA_PATH, sep="|", nrows=0).columns.tolist()
sample = pd.read_csv(DATA_PATH, sep="|", dtype="string", nrows=8)

print(f"Nombre de colonnes : {len(columns)}")
display(pd.Series(columns, name="Colonne").to_frame())
columns_to_show = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
]
display(sample[columns_to_show])

## 2. Profil global en lecture par blocs

Chaque ligne du fichier n’est pas nécessairement une vente distincte. On commence donc par décrire les lignes telles qu’elles sont fournies, sans leur attribuer prématurément une unité statistique. Le profil complet peut prendre quelques dizaines de secondes.

In [ ]:
row_count = 0
missing_counts = pd.Series(0, index=columns, dtype="int64")
local_type_counts = pd.Series(dtype="int64")
mutation_type_counts = pd.Series(dtype="int64")
price_categories = ["missing", "zero", "positive", "positive_under_10000"]
price_counts = pd.Series(0, index=price_categories, dtype="int64")
date_min = None
date_max = None

for chunk in pd.read_csv(DATA_PATH, sep="|", dtype="string", chunksize=100_000):
    row_count += len(chunk)
    missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0)
    local_type_counts = local_type_counts.add(
        chunk["Type local"].value_counts(dropna=False), fill_value=0
    )
    mutation_type_counts = mutation_type_counts.add(
        chunk["Nature mutation"].value_counts(dropna=False), fill_value=0
    )

    prices = pd.to_numeric(
        chunk["Valeur fonciere"].str.replace(",", ".", regex=False),
        errors="coerce",
    )
    price_counts += pd.Series(
        {
            "missing": int(prices.isna().sum()),
            "zero": int(prices.eq(0).sum()),
            "positive": int(prices.gt(0).sum()),
            "positive_under_10000": int((prices.gt(0) & prices.lt(10_000)).sum()),
        }
    )

    dates = pd.to_datetime(chunk["Date mutation"], format="%d/%m/%Y", errors="coerce")
    chunk_min, chunk_max = dates.min(), dates.max()
    date_min = chunk_min if date_min is None else min(date_min, chunk_min)
    date_max = chunk_max if date_max is None else max(date_max, chunk_max)

print(f"Lignes : {row_count:,}")
print(f"Période : {date_min:%Y-%m-%d} → {date_max:%Y-%m-%d}")
display(price_counts.to_frame("Nombre de lignes"))
local_type_summary = local_type_counts.sort_values(ascending=False).to_frame(
    "Lignes par type de local"
)
mutation_type_summary = mutation_type_counts.sort_values(ascending=False).to_frame(
    "Lignes par nature de mutation"
)
display(local_type_summary)
display(mutation_type_summary)

### Première lecture des valeurs manquantes

Une valeur manquante n’a pas le même sens pour toutes les colonnes. Par exemple, une surface bâtie absente peut être normale pour un terrain non bâti, alors qu’elle est importante pour estimer le prix d’une maison. On doit donc étudier les manquants après avoir défini le type de bien visé.

In [ ]:
missing_summary = (
    missing_counts.rename("Nombre manquant")
    .to_frame()
    .assign(pourcentage=lambda table: 100 * table["Nombre manquant"] / row_count)
    .sort_values("pourcentage", ascending=False)
)
display(missing_summary.head(20).round(1))

## 3. Pourquoi une ligne n’est pas une vente

La notice de la DGFiP indique qu’une disposition peut produire plusieurs lignes, notamment une ligne par local, et que le prix est répété sur ces lignes. Le numéro de disposition est défini dans le contexte d’un acte. Dans ce fichier, les champs d’identification/référence du document sont entièrement vides : `Date mutation + département + commune + No disposition` n’est donc pas une clé fiable à elle seule.

La cellule suivante cherche une démonstration dans les 200 000 premières lignes. Elle montre des groupes ayant la même clé candidate mais des prix distincts : ces groupes réunissent plusieurs dispositions/ventes et ne doivent pas être additionnés ou utilisés comme une vente sans investigation.

Référence : [notice descriptive officielle DVF](https://www.data.gouv.fr/api/1/datasets/r/d573456c-76eb-4276-b91c-e6b9c89d6656).

In [ ]:
candidate_columns = [
    "Date mutation",
    "Code departement",
    "Code commune",
    "No disposition",
]
first_rows = pd.read_csv(DATA_PATH, sep="|", dtype="string", nrows=200_000)
candidate_groups = first_rows.groupby(candidate_columns, dropna=False)
candidate_price_counts = candidate_groups["Valeur fonciere"].nunique(dropna=False)
ambiguous_keys = candidate_price_counts[candidate_price_counts > 1]
print(f"Groupes ambigus dans cet extrait : {len(ambiguous_keys):,}")

if not ambiguous_keys.empty:
    example_key = ambiguous_keys.index[0]
    example_mask = pd.Series(True, index=first_rows.index)
    for column, value in zip(candidate_columns, example_key, strict=True):
        example_mask &= first_rows[column].eq(value)
    display(
        first_rows.loc[
            example_mask,
            candidate_columns
            + [
                "Nature mutation",
                "Valeur fonciere",
                "Type local",
                "Surface reelle bati",
            ],
        ]
    )
else:
    print(
        "Aucun exemple ambigu dans cet extrait ; cela ne prouve pas "
        "que la clé est unique."
    )

## 4. Critères de base retenus

Pour la première cohorte d’analyse, on garde uniquement les lignes où `Nature mutation` vaut exactement `Vente`, le `Type local` est `Maison` ou `Appartement`, et le prix, la surface bâtie et le nombre de pièces sont renseignés. Ces règles portent sur les lignes du fichier ; elles ne suffisent pas encore à garantir une vente unique par observation.

On compte séparément les champs numériques illisibles et les valeurs nulles ou négatives. Ils ne sont pas silencieusement supprimés par ce filtre de présence. Un nombre de pièces égal à `0` est conservé dès lors que la surface bâtie est renseignée.

In [ ]:
eligible_by_type = pd.Series(dtype="int64")
quality_counts = pd.Series(
    0,
    index=[
        "eligible_rows",
        "price_unparseable",
        "surface_unparseable",
        "rooms_unparseable",
        "price_zero_or_negative",
        "surface_zero_or_negative",
        "rooms_zero_or_negative",
    ],
    dtype="int64",
)

required_columns = [
    "Valeur fonciere",
    "Surface reelle bati",
    "Nombre pieces principales",
]
for chunk in pd.read_csv(DATA_PATH, sep="|", dtype="string", chunksize=100_000):
    is_residential = chunk["Type local"].isin(["Maison", "Appartement"])
    has_required_values = chunk[required_columns].apply(
        lambda column: column.str.strip().ne("").fillna(False)
    ).all(axis=1)
    eligible = (
        chunk["Nature mutation"].eq("Vente")
        & is_residential
        & has_required_values
    )
    eligible_by_type = eligible_by_type.add(
        chunk.loc[eligible, "Type local"].value_counts(), fill_value=0
    )
    quality_counts["eligible_rows"] += int(eligible.sum())

    price = pd.to_numeric(
        chunk.loc[eligible, "Valeur fonciere"].str.replace(",", ".", regex=False),
        errors="coerce",
    )
    surface = pd.to_numeric(
        chunk.loc[eligible, "Surface reelle bati"], errors="coerce"
    )
    rooms = pd.to_numeric(
        chunk.loc[eligible, "Nombre pieces principales"], errors="coerce"
    )
    quality_counts["price_unparseable"] += int(price.isna().sum())
    quality_counts["surface_unparseable"] += int(surface.isna().sum())
    quality_counts["rooms_unparseable"] += int(rooms.isna().sum())
    quality_counts["price_zero_or_negative"] += int(price.le(0).sum())
    quality_counts["surface_zero_or_negative"] += int(surface.le(0).sum())
    quality_counts["rooms_zero_or_negative"] += int(rooms.le(0).sum())

display(eligible_by_type.astype("int64").to_frame("Lignes éligibles"))
display(quality_counts.to_frame("Nombre de lignes"))

## 5. Vérifier les identifiants et les doublons possibles

Avant de dédupliquer, on vérifie les identifiants fournis. Les colonnes `Identifiant de document`, `Reference document` et `Identifiant local` sont vides dans le fichier public. Le numéro de disposition seul ne suffit pas.

On calcule donc une **signature candidate** faite des références cadastrales, des lots, du type et des caractéristiques du logement, dans le contexte de la vente. Des lignes ayant la même signature sont des doublons possibles à examiner ; cette signature ne prouve pas qu’il s’agit du même logement, car plusieurs logements peuvent partager une parcelle et certaines références de lot sont absentes. On ne les supprimera pas automatiquement à cette étape.

In [ ]:
explicit_ids = [
    "Identifiant de document",
    "Reference document",
    "Identifiant local",
]
signature_columns = [
    "Date mutation",
    "Nature mutation",
    "No disposition",
    "Code departement",
    "Code commune",
    "Valeur fonciere",
    "Type local",
    "Prefixe de section",
    "Section",
    "No plan",
    "No Volume",
    "1er lot",
    "2eme lot",
    "3eme lot",
    "4eme lot",
    "5eme lot",
    "Surface reelle bati",
    "Nombre pieces principales",
]
audit_columns = list(dict.fromkeys(signature_columns + explicit_ids))
missing_ids = pd.Series(0, index=explicit_ids, dtype="int64")
signature_hashes = []
audited_rows = 0

for chunk in pd.read_csv(
    DATA_PATH, sep="|", usecols=audit_columns, dtype="string", chunksize=100_000
):
    required_values = [
        "Valeur fonciere",
        "Surface reelle bati",
        "Nombre pieces principales",
    ]
    has_required_values = chunk[required_values].apply(
        lambda column: column.str.strip().ne("").fillna(False)
    ).all(axis=1)
    eligible = (
        chunk["Nature mutation"].eq("Vente")
        & chunk["Type local"].isin(["Maison", "Appartement"])
        & has_required_values
    )
    selected = chunk.loc[eligible]
    audited_rows += len(selected)
    missing_ids += selected[explicit_ids].isna().sum()
    signature = selected[signature_columns].fillna("")
    signature_hashes.append(
        pd.util.hash_pandas_object(signature, index=False).to_numpy(dtype="uint64")
    )

all_signatures = np.concatenate(signature_hashes)
unique_signatures = len(np.unique(all_signatures))
print(f"Lignes examinées : {audited_rows:,}")
display(missing_ids.to_frame("Identifiants manquants"))
print(f"Signatures candidates distinctes : {unique_signatures:,}")
duplicate_signature_rows = audited_rows - unique_signatures
print(
    f"Lignes supplémentaires ayant une signature déjà vue : "
    f"{duplicate_signature_rows:,}"
)

## Conclusion de cette étape

- La cohorte de base est limitée aux ventes de maisons ou d’appartements avec prix, surface bâtie et pièces renseignés.
- Le prix est répété sur plusieurs lignes ; une observation de modèle ne peut pas être choisie simplement en gardant chaque ligne.
- Le fichier ne fournit pas d’identifiant stable du logement ou du document. Les signatures répétées sont des candidates à l’audit, pas des doublons certains.
- Les colonnes de localisation et de lots sont gardées pour cet audit ; seules les colonnes utiles aux caractéristiques, à la cible et à la déduplication entreront dans le jeu de travail.
- Les prix très faibles doivent être examinés comme cas particuliers, pas supprimés automatiquement.

La suite de l’exploration portera sur les groupes de mutations et les cas à conserver pour un premier jeu d’analyse. Aucun preprocessing ni entraînement n’est encore effectué.

> **Réutilisation des données :** respecter les [conditions officielles](https://www.data.gouv.fr/api/1/datasets/r/99549bdd-91f1-4a99-ac00-855b9a14e5f6), notamment l’interdiction de chercher à réidentifier des personnes et de faire indexer ces données par des moteurs externes.